# Model Deployment with Flask

## Notebook Overview

In this notebook, we focus on deploying our final machine learning model using Flask to make it accessible through a web application.

Up to this point in the project, we have already:
- Explored and understood the dataset
- Cleaned and prepared the data
- Engineered relevant features
- Built and evaluated machine learning models

This notebook represents the final stage of the machine learning workflow — taking the trained model and transforming it into a usable application that stakeholders and end users can interact with.

## Why Deployment Matters

Building a machine learning model is only part of the solution.  
For the model to create real-world value, it needs to be accessible and usable outside the notebook environment.

By deploying the model using Flask:
- Predictions can be generated in real time
- Users can interact with the model through a simple interface
- The solution becomes easier to demonstrate, test, and potentially integrate into real systems

This step bridges the gap between experimentation and practical application.


## What This Notebook Covers

This notebook walks through the deployment process by:

- Loading the trained machine learning model
- Preparing the application environment
- Creating prediction functions
- Building a Flask application
- Connecting user inputs to the model
- Generating predictions from new data
- Testing the application locally


## Project Objective

The deployed application is designed to support financial inclusion analysis by predicting financial inclusion risk based on user-provided information.

The aim is to provide a simple and efficient way for stakeholders to interact with the predictive model without needing technical knowledge of machine learning workflows.


## Expected Outcome

At the end of this notebook, we expect to have:
- A working Flask application
- A deployed machine learning prediction pipeline
- A functional interface capable of generating predictions from user input

This demonstrates how machine learning solutions can move beyond development and into practical use cases.

In [1]:
import json
import joblib
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path("..")

ARTIFACTS_DIR = BASE_DIR / "artifacts"
MODELS_DIR = BASE_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

MODEL_PATH = MODELS_DIR / "financial_exclusion_gradient_boosting_pipeline.joblib"
FEATURE_COLUMNS_PATH = METRICS_DIR / "04_feature_columns.json"
METADATA_PATH = METRICS_DIR / "04_model_metadata.json"
SAMPLE_INPUT_PATH = METRICS_DIR / "04_sample_input.csv"

print(MODEL_PATH.exists())
print(FEATURE_COLUMNS_PATH.exists())
print(METADATA_PATH.exists())
print(SAMPLE_INPUT_PATH.exists())

True
True
True
True


In [3]:
model = joblib.load(MODEL_PATH)

with open(FEATURE_COLUMNS_PATH, "r") as f:
    feature_columns = json.load(f)

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

sample_input = pd.read_csv(SAMPLE_INPUT_PATH)

print(type(model))
print("Number of expected features:", len(feature_columns))
print("Sample input shape:", sample_input.shape)
metadata

<class 'sklearn.pipeline.Pipeline'>
Number of expected features: 9
Sample input shape: (1, 9)


{'best_model': 'Gradient Boosting',
 'rows': 20902,
 'features': 9,
 'target_positive_rate': 0.17567696871112812,
 'mean_cv_roc_auc': 0.8176950419318123,
 'std_cv_roc_auc': 0.007642404456424021}

In [4]:
missing_cols = [col for col in feature_columns if col not in sample_input.columns]
extra_cols = [col for col in sample_input.columns if col not in feature_columns]

print("Missing columns:", missing_cols)
print("Extra columns:", extra_cols)

assert len(missing_cols) == 0, "Sample input is missing required columns."

sample_input = sample_input[feature_columns]

print("Sample input validated.")

Missing columns: []
Extra columns: []
Sample input validated.


In [5]:
prediction = model.predict(sample_input)
probability = model.predict_proba(sample_input)

print("Prediction:", prediction[0])
print("Probability of inclusion:", probability[0][0])
print("Probability of exclusion:", probability[0][1])

Prediction: 0
Probability of inclusion: 0.9639180604412639
Probability of exclusion: 0.036081939558736044


In [6]:
def assign_risk_tier(exclusion_probability):
    if exclusion_probability < 0.40:
        return "Low Risk"
    elif exclusion_probability < 0.70:
        return "Medium Risk"
    else:
        return "High Risk"

In [7]:
def predict_financial_exclusion(input_data):
    input_df = pd.DataFrame([input_data])
    input_df = input_df[feature_columns]

    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]
    risk_tier = assign_risk_tier(probability)

    result = {
        "prediction": int(prediction),
        "prediction_label": "Financially Excluded" if prediction == 1 else "Financially Included",
        "exclusion_probability": round(float(probability), 4),
        "exclusion_percentage": f"{probability * 100:.2f}%",
        "risk_tier": risk_tier
    }

    return result

In [8]:
sample_dict = sample_input.iloc[0].to_dict()

result = predict_financial_exclusion(sample_dict)
result

{'prediction': 0,
 'prediction_label': 'Financially Included',
 'exclusion_probability': 0.0361,
 'exclusion_percentage': '3.61%',
 'risk_tier': 'Low Risk'}

In [9]:
flask_app_code = '''
from flask import Flask, request, jsonify, render_template
import joblib
import json
import pandas as pd
from pathlib import Path

app = Flask(__name__)

BASE_DIR = Path(__file__).resolve().parent.parent

MODEL_PATH = BASE_DIR / "models" / "financial_exclusion_gradient_boosting_pipeline.joblib"
FEATURE_COLUMNS_PATH = BASE_DIR / "artifacts" / "metrics" / "04_feature_columns.json"

model = joblib.load(MODEL_PATH)

with open(FEATURE_COLUMNS_PATH, "r") as f:
    feature_columns = json.load(f)


def assign_risk_tier(exclusion_probability):
    if exclusion_probability < 0.40:
        return "Low Risk"
    elif exclusion_probability < 0.70:
        return "Medium Risk"
    else:
        return "High Risk"


def make_prediction(input_data):
    input_df = pd.DataFrame([input_data])

    # Add missing columns required by the trained pipeline
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = None

    # Keep exact column order used during training
    input_df = input_df[feature_columns]

    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]
    risk_tier = assign_risk_tier(probability)

    result = {
        "prediction": int(prediction),
        "prediction_label": "Financially Excluded" if prediction == 1 else "Financially Included",
        "exclusion_probability": round(float(probability), 4),
        "exclusion_percentage": f"{probability * 100:.2f}%",
        "risk_tier": risk_tier
    }

    return result


@app.route("/")
def home():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():
    try:
        input_data = request.get_json()
        result = make_prediction(input_data)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)}), 400


@app.route("/predict_form", methods=["POST"])
def predict_form():
    try:
        input_data = {
            "gender": request.form.get("gender"),
            "settlement_type": request.form.get("settlement_type"),
            "education_level": request.form.get("education_level"),
            "employment_status": request.form.get("employment_status")
        }

        result = make_prediction(input_data)

        return render_template("index.html", result=result)

    except Exception as e:
        return render_template("index.html", error=str(e))


if __name__ == "__main__":
    app.run(debug=True)
'''

print(flask_app_code)


from flask import Flask, request, jsonify, render_template
import joblib
import json
import pandas as pd
from pathlib import Path

app = Flask(__name__)

BASE_DIR = Path(__file__).resolve().parent.parent

MODEL_PATH = BASE_DIR / "models" / "financial_exclusion_gradient_boosting_pipeline.joblib"
FEATURE_COLUMNS_PATH = BASE_DIR / "artifacts" / "metrics" / "04_feature_columns.json"

model = joblib.load(MODEL_PATH)

with open(FEATURE_COLUMNS_PATH, "r") as f:
    feature_columns = json.load(f)


def assign_risk_tier(exclusion_probability):
    if exclusion_probability < 0.40:
        return "Low Risk"
    elif exclusion_probability < 0.70:
        return "Medium Risk"
    else:
        return "High Risk"


def make_prediction(input_data):
    input_df = pd.DataFrame([input_data])

    # Add missing columns required by the trained pipeline
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = None

    # Keep exact column order used during train

In [10]:
APP_DIR = BASE_DIR / "app"
APP_DIR.mkdir(exist_ok=True)

APP_PATH = APP_DIR / "app.py"

with open(APP_PATH, "w") as f:
    f.write(flask_app_code)

print(f"Flask app saved to: {APP_PATH}")

Flask app saved to: ..\app\app.py


# Deployment Notes

The Flask application loads the saved Gradient Boosting pipeline and feature column list.

The app exposes a `/predict` endpoint that accepts JSON input, applies the saved preprocessing pipeline, generates a financial exclusion prediction, and returns:

- predicted class,
- exclusion probability,
- percentage risk,
- and risk tier.

The app can be started locally using:

```bash
python app/app.py


This deployment is meant for demonstration purposes only

# Very Important

The Flask app will only work if the JSON input contains **all columns in `04_deployment_feature_columns.json`**.

After running this notebook, your project should have, models and artifacts are saved from the model and evaluation notebook:

```text
app/
└── app.py
models/
 └── financial_exclusion_gradient_boosting_pipeline.joblib
artifacts/
└── metrics/
    ├── 04_deployment_feature_columns.json
    ├── 04_deployment_metadata.json
    └── 04_sample_deployment_input.csv